# H&N vs Lung – 867 FASPRO purchases: final segment with CareSet split

**Goal:** give every recent 867 FASPRO purchase a **final segment** (Head & Neck / Lung / Unclassified) and a **reason** explaining why.

**Inputs**
| Dataset | Content |
|---|---|
| `867_Recent_Week_Purchase_L2L3_Segmentation` | Recent 867 FASPRO purchases, rolled from L1 to L2/L3 (`L2_L3`) and joined to the historical account segmentation (`segment`) |
| `Careset_L2_L3_Split` | CareSet (Medicare) account-level patient split: `%Lung Indication`, `%HNSCC Indication`, `%Both Indication` |

**Logic**
1. **H&N only / Lung only** purchases keep their historical segment.
2. **H&N + Lung**, or no segment: use the account's CareSet shares.
   - **H&N share** = %HNSCC + %Both × %HNSCC / (%HNSCC + %Lung)
   - **Lung share** = %Lung + %Both × %Lung / (%HNSCC + %Lung)
   - H&N share ≥ H&N threshold → **Head & Neck**; Lung share ≥ Lung threshold → **Lung**; otherwise **Unclassified**.
3. Purchases with no L2/L3 mapping are **kept**, as Unclassified, with the reason stated.

**Steps:** 1 setup · 2 CareSet shares · 3 averages (for the threshold discussion) · 4 apply threshold to 867 · 5 save

## 1. Setup

### What this cell does: settings, helpers and loading
- Dataset names, the two thresholds (left empty until agreed with Ravi), and the output name.
- Helpers:
  - `clean()`: blanks and placeholders like `"-"` become null;
  - `pick()`: finds a column despite spaces or symbols in its name (e.g. `%Lung Indication`).
- Loads both datasets and prints their row counts.

In [ ]:
import re
import dataiku
import dataiku.spark as dkuspark
from pyspark.sql import SparkSession, SQLContext, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
sqlContext = SQLContext(spark.sparkContext)

# ---------------- settings ----------------
CARESET_SPLIT = "Careset_L2_L3_Split"
DS_867 = "867_Recent_Week_Purchase_L2L3_Segmentation"
OUT = "867_Recent_Week_Purchase_Final_Segment"

HN_SHARE_THRESHOLD = None     # set after agreement, e.g. 0.50 -> Head & Neck
LUNG_SHARE_THRESHOLD = None   # set after agreement, e.g. 0.85 -> Lung
SAVE = False
# -------------------------------------------

MISSING = ["", "-", "--", "NA", "N/A", "NULL", "NONE", "NAN"]

def load(name):
    return dkuspark.get_dataframe(sqlContext, dataiku.Dataset(name))

def clean(col):
    s = F.trim(col.cast("string"))
    return F.when(~F.upper(s).isin(MISSING), s)

def pick(d, name):
    key = lambda x: re.sub(r"[^0-9a-z]", "", x.lower())
    found = {key(x): x for x in d.columns}
    if key(name) not in found:
        raise KeyError(f"column '{name}' not found. Available: {d.columns}")
    return F.col(f"`{found[key(name)]}`")

raw = load(CARESET_SPLIT)
s867 = load(DS_867)
print(f"{CARESET_SPLIT}: {raw.count():,} rows | {DS_867}: {s867.count():,} rows")

## 2. CareSet shares per account

### What this cell does: one row per CareSet account with its Lung and H&N share
- Reads `PARENT_ID` (the L2/L3 account), `opal_lvl`, the flags, the patient counts and the three % columns.
- **One row per account:** exact duplicates are dropped. If an account still appears more than once, the row with the most patients is kept.
- **Shares** (team formula): `%Both` is split in proportion to `%HNSCC` vs `%Lung`.
  - Accounts with only `%Both` (or no %) get no share.
  - The two shares add up to 1.
- **Checks:**
  - accounts without a share;
  - rows where the three % don't add up to 1;
  - accounts with an H&N share of exactly 0, and how many of those are small (≤ 20 patients). CareSet hides counts below 11, so a 0 may really be a small hidden number.

In [ ]:
num = lambda n: clean(pick(raw, n)).cast("double")
cs = raw.select(clean(pick(raw, "PARENT_ID")).alias("account_id"),
                clean(pick(raw, "opal_lvl")).alias("cs_level"),
                clean(pick(raw, "Priority Account flag")).alias("cs_priority_flag"),
                clean(pick(raw, "total_patients_BIMS_flag")).alias("cs_bims_flag"),
                num("total_lung+HNSCC").alias("cs_total_pts"),
                num("total_lung_patients").alias("cs_lung_pts"),
                num("total_hnscc_patients").alias("cs_hnscc_pts"),
                num("total_both_patients").alias("cs_both_pts"),
                num("%Lung Indication").alias("pct_lung"),
                num("%HNSCC Indication").alias("pct_hn"),
                num("%Both Indication").alias("pct_both")).filter("account_id IS NOT NULL")

n0 = cs.count()
cs = cs.dropDuplicates()
cs = (cs.withColumn("rk", F.row_number().over(Window.partitionBy("account_id")
                                              .orderBy(F.desc_nulls_last("cs_total_pts"), "cs_level")))
        .filter("rk = 1").drop("rk"))
print(f"CareSet rows {n0:,} -> distinct accounts {cs.count():,}")

den = F.col("pct_hn") + F.col("pct_lung")
cs = (cs.withColumn("hn_share", F.when(den > 0, F.col("pct_hn") + F.col("pct_both") * F.col("pct_hn") / den))
        .withColumn("lung_share", F.when(den > 0, F.col("pct_lung") + F.col("pct_both") * F.col("pct_lung") / den))
        .cache())

print("Checks:")
cs.agg(F.sum(F.col("hn_share").isNull().cast("int")).alias("no_share_(only_Both_or_missing)"),
       F.sum((F.abs(F.col("pct_lung") + F.col("pct_hn") + F.col("pct_both") - 1) > 0.01).cast("int")).alias("pcts_not_summing_to_1"),
       F.sum((F.col("hn_share") == 0).cast("int")).alias("accounts_hn_share_0"),
       F.sum(((F.col("hn_share") == 0) & (F.col("cs_total_pts") <= 20)).cast("int")).alias("hn_0_and_total_<=20_(suppression?)")
       ).show(truncate=False)
cs.select("account_id", "cs_level", "cs_total_pts", "pct_lung", "pct_hn", "pct_both", "hn_share", "lung_share").show(10, False)

## 3. Average shares (for the threshold discussion)

### What this cell does: the numbers to share with Ravi
- For each group of accounts it shows:
  - the number of accounts;
  - **simple average** H&N and Lung share (every account counts the same);
  - **patient-weighted average** (larger accounts count more; recommended);
  - **median** H&N share.
- **Groups:**
  - **A) All CareSet accounts**, plus a split by level (L2 / L3);
  - **B) Only the accounts behind ambiguous 867 purchases** (segment H&N + Lung or no segment): the accounts the threshold will actually be applied to.
- **Distribution** of the H&N share in bands (0, 0–10%, …, ≥ 50%).

In [ ]:
def averages(d, label):
    print(label)
    d.filter("hn_share IS NOT NULL").agg(
        F.count("*").alias("accounts"),
        F.round(F.avg("hn_share"), 4).alias("avg_hn_share"),
        F.round(F.avg("lung_share"), 4).alias("avg_lung_share"),
        F.round(F.sum(F.col("hn_share") * F.col("cs_total_pts")) / F.sum("cs_total_pts"), 4).alias("wtd_hn_share"),
        F.round(F.sum(F.col("lung_share") * F.col("cs_total_pts")) / F.sum("cs_total_pts"), 4).alias("wtd_lung_share"),
        F.round(F.expr("percentile_approx(hn_share, 0.5)"), 4).alias("median_hn_share")).show(truncate=False)

averages(cs, "A) All CareSet accounts")
for lvl in sorted([r[0] for r in cs.select("cs_level").distinct().collect() if r[0] is not None]):
    averages(cs.filter(F.col("cs_level") == lvl), f"   level {lvl}")

amb = (s867.select(clean(pick(s867, "L2_L3")).alias("account_id"), clean(pick(s867, "segment")).alias("segment"))
           .filter(F.col("account_id").isNotNull() &
                   ~F.coalesce(F.col("segment"), F.lit("")).isin("H&N only", "Lung only"))
           .select("account_id").distinct())
print("Ambiguous 867 accounts:", amb.count(), "| with a CareSet split:", amb.join(cs, "account_id").count())
averages(cs.join(amb, "account_id"), "B) Ambiguous 867 accounts only")

print("Distribution of H&N share (all CareSet accounts):")
(cs.filter("hn_share IS NOT NULL")
   .withColumn("hn_share_band", F.when(F.col("hn_share") == 0, "a) 0")
               .when(F.col("hn_share") < 0.1, "b) 0-10%").when(F.col("hn_share") < 0.2, "c) 10-20%")
               .when(F.col("hn_share") < 0.3, "d) 20-30%").when(F.col("hn_share") < 0.5, "e) 30-50%")
               .otherwise("f) >=50%"))
   .groupBy("hn_share_band").count().orderBy("hn_share_band").show())

## 4. Apply the threshold to the 867 purchases

### What this cell does: final segment and reason for every 867 record
- **Runs only after both thresholds are set** in section 1.
  - Their sum must be greater than 1, so a purchase can never qualify for both.
- Joins each 867 record to its account's CareSet shares on `L2_L3` = `PARENT_ID`.
  - Checks that the number of records doesn't change.
- **Final segment:**

| 867 record | final_segment | classification_source |
|---|---|---|
| No L2/L3 mapping | Unclassified (kept) | NONE |
| Segment H&N only / Lung only | Head & Neck / Lung | SEGMENTATION |
| H&N + Lung or no segment, no CareSet split | Unclassified | NONE |
| … H&N share ≥ H&N threshold | Head & Neck | CARESET |
| … Lung share ≥ Lung threshold | Lung | CARESET |
| … shares between the thresholds | Unclassified | CARESET |

- `classification_reason` says why in words, including the share and the threshold used.
- Prints records and quantity per final segment, and the count by source.

In [ ]:
assert HN_SHARE_THRESHOLD is not None and LUNG_SHARE_THRESHOLD is not None, \
    "Set HN_SHARE_THRESHOLD and LUNG_SHARE_THRESHOLD in section 1 first."
assert HN_SHARE_THRESHOLD + LUNG_SHARE_THRESHOLD > 1, "Thresholds overlap: a purchase could qualify for both."

d = (s867.withColumn("_L2_L3", clean(pick(s867, "L2_L3")))
         .withColumn("_segment", clean(pick(s867, "segment"))))
n_in = d.count()
d = d.join(cs.select(F.col("account_id").alias("_L2_L3"), "hn_share", "lung_share", "cs_total_pts", "cs_bims_flag"),
           "_L2_L3", "left")
assert d.count() == n_in, "Join changed the number of 867 records"

fmt = lambda c: F.format_number(F.col(c), 2)
seg, hn, lung = F.col("_segment"), F.col("hn_share"), F.col("lung_share")
seg_txt = F.coalesce(seg, F.lit("no segment"))
d = (d.withColumn("final_segment",
        F.when(F.col("_L2_L3").isNull(), "Unclassified")
         .when(seg == "H&N only", "Head & Neck")
         .when(seg == "Lung only", "Lung")
         .when(hn.isNull(), "Unclassified")
         .when(hn >= HN_SHARE_THRESHOLD, "Head & Neck")
         .when(lung >= LUNG_SHARE_THRESHOLD, "Lung")
         .otherwise("Unclassified"))
      .withColumn("classification_source",
        F.when(F.col("_L2_L3").isNull(), "NONE")
         .when(seg.isin("H&N only", "Lung only"), "SEGMENTATION")
         .when(hn.isNull(), "NONE").otherwise("CARESET"))
      .withColumn("classification_reason",
        F.when(F.col("_L2_L3").isNull(), F.lit("No L2/L3 mapping for the purchasing account"))
         .when(seg.isin("H&N only", "Lung only"), F.concat(F.lit("Historical segmentation: "), seg))
         .when(hn.isNull(), F.concat(F.lit("Segmentation '"), seg_txt, F.lit("'; no CareSet split for the account")))
         .when(hn >= HN_SHARE_THRESHOLD, F.concat(F.lit("Segmentation '"), seg_txt, F.lit("'; CareSet H&N share "),
                                                  fmt("hn_share"), F.lit(f" >= {HN_SHARE_THRESHOLD}")))
         .when(lung >= LUNG_SHARE_THRESHOLD, F.concat(F.lit("Segmentation '"), seg_txt, F.lit("'; CareSet Lung share "),
                                                      fmt("lung_share"), F.lit(f" >= {LUNG_SHARE_THRESHOLD}")))
         .otherwise(F.concat(F.lit("Segmentation '"), seg_txt, F.lit("'; CareSet shares between thresholds (H&N "),
                             fmt("hn_share"), F.lit(", Lung "), fmt("lung_share"), F.lit(")"))))
      .drop("_L2_L3", "_segment").cache())

qty = pick(d, "qty_sold_pu")
print(f"867 records in {n_in:,} | out {d.count():,}")
(d.groupBy("final_segment").agg(F.count("*").alias("records"), F.round(F.sum(qty), 1).alias("qty"))
  .withColumn("pct_qty", F.round(100 * F.col("qty") / F.sum("qty").over(Window.partitionBy()), 1))
  .orderBy("final_segment").show())
d.groupBy("classification_source", "final_segment").count().orderBy("classification_source", "final_segment").show(truncate=False)
d.groupBy("classification_reason").count().orderBy(F.desc("count")).show(30, False)

## 5. Save to the Flow

### What this cell does: write the final 867 table
- Set `SAVE = True` in section 1 to write.
- The output dataset is created automatically if it doesn't exist, on the same connection as the 867 input.
- **Output:** `867_Recent_Week_Purchase_Final_Segment`
  - every 867 record;
  - plus `hn_share`, `lung_share`, `cs_total_pts`, `cs_bims_flag`, `final_segment`, `classification_source`, `classification_reason`.

In [ ]:
if SAVE:
    client = dataiku.api_client()
    project = client.get_default_project()
    if OUT not in [x["name"] for x in project.list_datasets()]:
        conn = project.get_dataset(DS_867).get_settings().get_raw()["params"]["connection"]
        b = project.new_managed_dataset(OUT)
        b.with_store_into(conn)
        b.create()
        print("created", OUT)
    dkuspark.write_with_schema(dataiku.Dataset(OUT), d)
    print("saved", OUT)
else:
    print("SAVE = False: nothing written.")